<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/_Kids_Elementary_Numberline_Race.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson: The Number Line Race to Zero

### Overview for Educators and Parents
This notebook generates a high-quality educational animation designed to introduce elementary-level learners to the concept of **integers and absolute value**.

By framing the number line as a 'race track' where numbers move from their starting positions toward zero, students can visually grasp that positive and negative numbers are mirror images of each other.

---

### Learning Objectives
- **Understanding Integers:** Recognize that negative numbers exist to the left of zero and positive numbers to the right.
- **Distance from Zero:** Visualize that both -7 and +7 are exactly 7 steps away from zero.
- **Introduction to Absolute Value:** Conceptually understand that the 'magnitude' (distance) is what determines who reaches zero first in a fair race.
- **Symmetry:** Observe the mirrored relationship of numbers across the origin (0).

### Video Breakdown
1.  **The Hook:** A suspenseful opening asking "Who reaches zero first?"
2.  **Introduction:** Meeting the 'Red Marble' (-7) and 'Blue Marble' (+7).
3.  **The Race:** A side-by-side movement tracking their positions and distances in real-time.
4.  **The Result:** A photo-finish 'Tie' to emphasize equal distance.
5.  **The Expert Panel:** A brief explanation of **Absolute Value** (|x|) for advanced learners.

---

### How to use this Notebook
- **Customization:** You can edit the CONFIG block in the code cell below to change colors, starting positions (e.g., race -5 vs +10), or even the text.
- **Generation:** Run the main code cell to render the video file.
- **Download:** Use the final cell to view and save the video for use in classrooms or presentations.

In [3]:
"""
Author:    Mugambi Ndwiga
Instagram: @craftsandengineering
GitHub:    github.com/zombimann/Mathematical-video-animations-and-visualization
Level:     Elementary
Concept:   Negative Numbers – Number Line Race to Zero

Description
-----------
A visually engaging, production-quality educational animation for young learners.
A red marble (negative number, starts at -7) and a blue marble (positive number,
starts at +7) race along a number line toward zero.  The video uses story-telling
hooks, sequencing and suspense to introduce the concept that negative and positive
numbers are mirror-images about zero on the number line.

Parametric controls (edit the CONFIG block below to experiment):
  - Number line range, dot starting positions, colours, speed, frame-rate, etc.
"""

# ──────────────────────────────────────────────────────────────────────────────
# PARAMETRIC CONFIG
# ──────────────────────────────────────────────────────────────────────────────
CONFIG = dict(
    FRAME_W           = 1280,
    FRAME_H           = 720,
    FPS               = 24,
    BG_CREAM          = (253, 248, 231, 255),
    GRID_GREY         = (180, 180, 180, 20),
    RED_DOT           = (255, 107, 107, 255),
    BLUE_DOT          = ( 77, 150, 255, 255),
    YELLOW_HL         = (255, 217,  61, 255),
    GREEN_HL          = (107, 203, 119, 255),
    INK               = ( 45,  45,  45, 255),
    WHITE             = (255, 255, 255, 255),
    ZERO_GOLD         = (255, 200,  40, 255),
    AXIS_GREY         = (140, 140, 140, 255),
    WATERMARK_ALPHA   = 153,
    LABEL_BG          = ( 77, 150, 255, 204),
    NL_LEFT           = -10,
    NL_RIGHT          =  10,
    RED_START         =  -7,
    BLUE_START        =   7,
    DOT_RADIUS        = 28,
    TICK_HEIGHT       = 20,
    FONT_BOLD         = "/usr/share/fonts/truetype/google-fonts/Poppins-Bold.ttf",
    FONT_REG          = "/usr/share/fonts/truetype/google-fonts/Poppins-Regular.ttf",
    FONT_MED          = "/usr/share/fonts/truetype/google-fonts/Poppins-Medium.ttf",
    OUTPUT_PATH       = "/mnt/user-data/outputs/number_line_race.mp4",
    QUALITY_CRF       = 22,
)

import os, math, subprocess
import numpy as np
from PIL import Image, ImageDraw, ImageFont

C = CONFIG
W, H, FPS = C["FRAME_W"], C["FRAME_H"], C["FPS"]
NL_Y  = int(H * 0.58)
NL_X0 = int(W * 0.08)
NL_X1 = int(W * 0.92)
NL_SPAN = C["NL_RIGHT"] - C["NL_LEFT"]

# ── helpers ────────────────────────────────────────────────

def ease(t):       return 3*t*t - 2*t*t*t
def ease_out(t):
    if t < 1/2.75:   return 7.5625*t*t
    elif t < 2/2.75: t -= 1.5/2.75;  return 7.5625*t*t + 0.75
    elif t < 2.5/2.75:t -= 2.25/2.75;return 7.5625*t*t + 0.9375
    else:            t -= 2.625/2.75;return 7.5625*t*t + 0.984375

def lerp(a, b, t): return a + (b-a)*t
def clamp(x):      return max(0.0, min(1.0, x))

def nl_to_px(v):
    frac = (v - C["NL_LEFT"]) / NL_SPAN
    return int(NL_X0 + frac*(NL_X1-NL_X0))

def load_font(path, size):
    try:    return ImageFont.truetype(path, size)
    except: return ImageFont.load_default()

F = {
    "title"   : load_font(C["FONT_BOLD"], 220),
    "sub"     : load_font(C["FONT_BOLD"], 150),
    "hook"    : load_font(C["FONT_BOLD"], 190),
    "label"   : load_font(C["FONT_BOLD"], 110),
    "annot"   : load_font(C["FONT_MED"],  96),
    "small"   : load_font(C["FONT_MED"],  84),
    "tiny"    : load_font(C["FONT_REG"],  64),
    "wm"      : load_font(C["FONT_MED"],  64),
    "card"    : load_font(C["FONT_BOLD"], 104),
    "card_s"  : load_font(C["FONT_MED"],  80),
    "num"     : load_font(C["FONT_BOLD"], 96),
    "closing" : load_font(C["FONT_BOLD"], 160),
    "closing_s": load_font(C["FONT_MED"], 110),
}

# ── drawing primitives ─────────────────────────────────────────────

def bg(img):
    d = ImageDraw.Draw(img)
    for x in range(0, W, 20): d.line([(x,0),(x,H)], fill=C["GRID_GREY"], width=1)
    for y in range(0, H, 20): d.line([(0,y),(W,y)], fill=C["GRID_GREY"], width=1)

def number_line(img, alpha=255, glow_zero=True):
    d = ImageDraw.Draw(img)
    ac = (140,140,140,alpha)
    d.line([(NL_X0,NL_Y),(NL_X1,NL_Y)], fill=ac, width=4)
    for v in range(C["NL_LEFT"], C["NL_RIGHT"]+1):
        px = nl_to_px(v)
        h2 = (C["TICK_HEIGHT"]+8)//2 if v==0 else C["TICK_HEIGHT"]//2
        d.line([(px,NL_Y-h2),(px,NL_Y+h2)], fill=ac, width=3 if v==0 else 2)
        lbl = str(v)
        bb = d.textbbox((0,0), lbl, font=F["num"])
        tw,th = bb[2]-bb[0], bb[3]-bb[1]
        lx,ly = px-tw//2, NL_Y+C["TICK_HEIGHT"]//2+6
        d.text((lx+1,ly+1), lbl, font=F["num"], fill=(255,255,255,min(200,alpha)))
        d.text((lx,ly), lbl, font=F["num"], fill=(45,45,45,alpha))
    if glow_zero:
        zx = nl_to_px(0)
        for r in range(24,0,-4):
            d.ellipse([zx-r,NL_Y-r,zx+r,NL_Y+r],
                      fill=(255,200,40, int(alpha*0.4*(24-r)/24)))

def marble(img, val, col, label="", glow=False, bounce=0):
    d = ImageDraw.Draw(img)
    r = C["DOT_RADIUS"]
    px, py = nl_to_px(val), NL_Y - r - 18 + bounce
    if glow:
        for gr in range(r+20, r, -4):
            d.ellipse([px-gr,py-gr,px+gr,py+gr],
                      fill=tuple(list(col[:3])+[int(40*(1-(gr-r)/20))]))
    d.ellipse([px-r+3,py-r+4,px+r+3,py+r+4], fill=(0,0,0,45))
    d.ellipse([px-r,py-r,px+r,py+r], fill=col, outline=(255,255,255,255), width=3)
    d.ellipse([px-r//3-2,py-r//3-2,px-r//3+6,py-r//3+6], fill=(255,255,255,150))
    if label:
        bb = d.textbbox((0,0), label, font=F["label"])
        tw,th = bb[2]-bb[0],bb[3]-bb[1]
        d.text((px-tw//2+1,py-th//2+1), label, font=F["label"], fill=(0,0,0,80))
        d.text((px-tw//2,  py-th//2),   label, font=F["label"], fill=(255,255,255,255))

def card(img, x, y, w, h, title, lines, bg_col=(255,255,255,220), accent=None):
    d = ImageDraw.Draw(img)
    d.rounded_rectangle([x+3,y+3,x+w+3,y+h+3], radius=16, fill=(0,0,0,35))
    d.rounded_rectangle([x,y,x+w,y+h], radius=16, fill=bg_col)
    if accent:
        d.rounded_rectangle([x,y,x+w,y+30], radius=16, fill=accent)
        d.rectangle([x,y+16,x+w,y+30], fill=accent)
    d.text((x+14,y+5), title, font=F["card"],
           fill=(255,255,255,255) if accent else (45,45,45,255))
    for i,ln in enumerate(lines):
        d.text((x+14,y+38+i*24), ln, font=F["card_s"], fill=(45,45,45,255))

def top_label(img, alpha=255):
    d = ImageDraw.Draw(img)
    txt = "For Kids: Elementary"
    bb  = d.textbbox((0,0), txt, font=F["small"])
    tw  = bb[2]-bb[0]; th = bb[3]-bb[1]
    pad = 12
    x0,y0 = W-tw-pad*2-16, 14
    x1,y1 = W-16, y0+th+pad
    bg_col = tuple(list(C["LABEL_BG"][:3])+[int(204*alpha/255)])
    d.rounded_rectangle([x0,y0,x1,y1], radius=10, fill=bg_col)
    d.text((x0+pad,y0+pad//2), txt, font=F["small"],
           fill=(255,255,255,alpha))

def watermark(img):
    d = ImageDraw.Draw(img)
    a = C["WATERMARK_ALPHA"]
    for i,ln in enumerate([") Mugambi Ndwiga","@craftsandengineering"]):
        bb = d.textbbox((0,0), ln, font=F["wm"])
        tw = bb[2]-bb[0]
        x,y = W-tw-14, H-50+i*22
        d.text((x+1,y+1), ln, font=F["wm"], fill=(255,255,255,int(a*0.7)))
        d.text((x,y), ln, font=F["wm"], fill=(80,80,80,a))

def centred(img, txt, y, fkey="sub", col=(45,45,45,255), alpha=255):
    d = ImageDraw.Draw(img)
    f = F[fkey]
    a = tuple(list(col[:3])+[alpha])
    bb = d.textbbox((0,0), txt, font=f)
    tx = (W-(bb[2]-bb[0]))//2
    d.text((tx+2,y+2), txt, font=f, fill=(255,255,255,min(alpha,200)))
    d.text((tx,y), txt, font=f, fill=a)

def new_frame():
    img = Image.new("RGBA", (W,H), C["BG_CREAM"])
    bg(img)
    return img

# ── scene generators (yield frames one at a time) ────────────────────────────────

def gen_hook(n):
    half = n//2
    for i in range(n):
        img = new_frame()
        d   = ImageDraw.Draw(img)
        t   = i/n
        if i < half:
            lt = i/half
            a  = int(255*ease(lt))
            pulse = 1.0 + 0.06*math.sin(lt*math.pi*4)
            qf = load_font(C["FONT_BOLD"], int(240*pulse))
            bb = d.textbbox((0,0),"?",font=qf)
            tw,th = bb[2]-bb[0],bb[3]-bb[1]
            qx,qy = (W-tw)//2,(H-th)//2-40
            d.text((qx+3,qy+3),"?",font=qf,fill=(255,255,255,100))
            d.text((qx,qy),"?",font=qf,fill=tuple(list(C["YELLOW_HL"][:3])+[a]))
            if lt > 0.35:
                a2 = int(255*ease((lt-0.35)/0.65))
                centred(img, "A question for you...", H//2+70, "sub", alpha=a2)
        else:
            lt = (i-half)/half
            a  = int(255*ease(min(1,lt*2)))
            centred(img, "Who reaches", H//2-70, "hook", alpha=a)
            if lt > 0.3:
                a2 = int(255*ease((lt-0.3)/0.7))
                zf = load_font(C["FONT_BOLD"], 144)
                zt = "ZERO"
                bb = d.textbbox((0,0),zt,font=zf)
                tw = bb[2]-bb[0]
                zx,zy = (W-tw)//2, H//2+10
                for gr in range(20,0,-5):
                    d.text((zx,zy),zt,font=zf,
                           fill=tuple(list(C["YELLOW_HL"][:3])+[int(a2*0.3*gr/20)]))
                d.text((zx+2,zy+2),zt,font=zf,fill=(255,255,255,a2//2))
                d.text((zx,zy),zt,font=zf,
                       fill=tuple(list(C["YELLOW_HL"][:3])+[a2]))
            if lt > 0.6:
                a3 = int(255*ease((lt-0.6)/0.4))
                centred(img, "first?", H//2+100, "hook", alpha=a3)
        top_label(img); watermark(img)
        yield img

def gen_intro(n):
    for i in range(n):
        img = new_frame()
        d   = ImageDraw.Draw(img)
        t   = i/n
        a_nl = int(255*ease(min(1,t*2)))
        number_line(img, alpha=a_nl)
        centred(img, "Meet our racers!", 28, "sub", alpha=int(255*ease(min(1,t*3))))
        slide = ease(min(1,t*2.5))
        ol = int((1-slide)*-320)
        or_ = int((1-slide)*320)
        cw,ch = 275,195
        # left card
        card(img, int(W*0.08)+ol, 100, cw, ch,
             "Red Marble",
             ["Negative number","Starts at  -7","Moves right ->","Toward ZERO"],
             bg_col=(255,235,235,220), accent=C["RED_DOT"])
        # right card
        card(img, W-cw-int(W*0.08)+or_, 100, cw, ch,
             "Blue Marble",
             ["Positive number","Starts at  +7","Moves left <-","Toward ZERO"],
             bg_col=(230,240,255,220), accent=C["BLUE_DOT"])
        # dots
        da = int(255*ease(min(1,max(0,t*3-1))))
        if da > 0:
            ov = Image.new("RGBA",(W,H),(0,0,0,0))
            marble(ov, C["RED_START"],  C["RED_DOT"],  "-7")
            marble(ov, C["BLUE_START"], C["BLUE_DOT"], "+7")
            arr = np.array(ov)
            arr[:,:,3] = (arr[:,:,3].astype(np.uint16)*da//255).astype(np.uint8)
            img.alpha_composite(Image.fromarray(arr,"RGBA"))
        if t > 0.72:
            a4 = int(255*ease((t-0.72)/0.28))
            d.text((NL_X0, NL_Y+54), "<- negative side",
                   font=F["annot"], fill=tuple(list(C["RED_DOT"][:3])+[a4]))
            d.text((W//2+18, NL_Y+54), "positive side ->",
                   font=F["annot"], fill=tuple(list(C["BLUE_DOT"][:3])+[a4]))
        top_label(img); watermark(img)
        yield img

def gen_explain(n):
    for i in range(n):
        img = new_frame()
        d   = ImageDraw.Draw(img)
        t   = i/n
        number_line(img, glow_zero=True)
        marble(img, C["RED_START"],  C["RED_DOT"],  "-7")
        marble(img, C["BLUE_START"], C["BLUE_DOT"], "+7")
        centred(img,"Numbers live on a line!", 28, "sub")
        # negative bubble
        a1 = int(255*ease(min(1,t*3)))
        lx = nl_to_px(C["RED_START"])
        if a1>10:
            bw,bh,bx,by = 225,76,lx-112,NL_Y-180
            d.rounded_rectangle([bx,by,bx+bw,by+bh],radius=13,
                fill=(255,235,235,a1),outline=tuple(list(C["RED_DOT"][:3])+[a1]),width=3)
            d.text((bx+12,by+8),"Negative",font=F["card"],fill=tuple(list(C["RED_DOT"][:3])+[a1]))
            d.text((bx+12,by+40),"less than zero",font=F["card_s"],fill=(45,45,45,a1))
            d.line([(lx,by+bh),(lx,NL_Y-C["DOT_RADIUS"]-22)],
                   fill=tuple(list(C["RED_DOT"][:3])+[a1]),width=2)
        # positive bubble
        a2 = int(255*ease(min(1,max(0,t*3-0.4))))
        rx = nl_to_px(C["BLUE_START"])
        if a2>10:
            bw2,bh2,bx2,by2 = 235,76,rx-117,NL_Y-180
            d.rounded_rectangle([bx2,by2,bx2+bw2,by2+bh2],radius=13,
                fill=(230,240,255,a2),outline=tuple(list(C["BLUE_DOT"][:3])+[a2]),width=3)
            d.text((bx2+12,by2+8),"Positive",font=F["card"],fill=tuple(list(C["BLUE_DOT"][:3])+[a2]))
            d.text((bx2+12,by2+40),"more than zero",font=F["card_s"],fill=(45,45,45,a2))
            d.line([(rx,by2+bh2),(rx,NL_Y-C["DOT_RADIUS"]-22)],
                   fill=tuple(list(C["BLUE_DOT"][:3])+[a2]),width=2)
        # symmetry bar
        if t>0.65:
            a3 = int(255*ease((t-0.65)/0.35))
            zx = nl_to_px(0)
            sx = nl_to_px(C["RED_START"]); ex = nl_to_px(C["BLUE_START"])
            sy = NL_Y+52
            d.line([(sx,sy),(ex,sy)],fill=tuple(list(C["YELLOW_HL"][:3])+[a3]),width=3)
            d.text((zx-80,sy+8),"<- same distance ->",
                   font=F["tiny"],fill=(45,45,45,a3))
        top_label(img); watermark(img)
        yield img

def gen_countdown(n):
    stages = ["3","2","1","GO!"]
    sf = n//len(stages)
    for i in range(n):
        img = new_frame()
        d   = ImageDraw.Draw(img)
        stage = min(i//sf, len(stages)-1)
        lt    = (i % sf)/sf
        et    = ease_out(lt)
        lbl   = stages[stage]
        is_go = stage==3
        sz = int(lerp(280,200,et)) if not is_go else int(lerp(160,220,et))
        cf = load_font(C["FONT_BOLD"], sz)
        bb = d.textbbox((0,0),lbl,font=cf)
        tw,th = bb[2]-bb[0],bb[3]-bb[1]
        tx,ty = (W-tw)//2,(H-th)//2
        col = [C["RED_DOT"],C["BLUE_DOT"],C["RED_DOT"],C["GREEN_HL"]][stage]
        for gr in range(80,0,-16):
            d.ellipse([W//2-gr,H//2-gr,W//2+gr,H//2+gr],
                      fill=tuple(list(col[:3])+[int(18*(1-gr/80))]))
        d.text((tx+3,ty+3),lbl,font=cf,fill=(255,255,255,120))
        d.text((tx,ty),lbl,font=cf,fill=col)
        number_line(img,alpha=70,glow_zero=False)
        marble(img, C["RED_START"], C["RED_DOT"])
        marble(img, C["BLUE_START"],C["BLUE_DOT"])
        top_label(img); watermark(img)
        yield img

def gen_race(n):
    race_end = int(n*0.76)
    rng = np.random.default_rng(42)
    nc  = 70
    cx  = rng.uniform(NL_X0, NL_X1, nc)
    cy  = rng.uniform(H//3,  H*0.88, nc)
    cdx = rng.uniform(-3,3, nc); cdy = rng.uniform(-7,-1,nc)
    ccols = [C["RED_DOT"],C["BLUE_DOT"],C["YELLOW_HL"],C["GREEN_HL"]]
    cci  = rng.integers(0,4,nc)
    cr   = rng.uniform(4,10,nc)

    for i in range(n):
        img = new_frame()
        d   = ImageDraw.Draw(img)
        t   = i/n

        # positions
        pt  = clamp(i/race_end)
        rp  = lerp(float(C["RED_START"]),  0.0, ease(pt))
        bp  = lerp(float(C["BLUE_START"]), 0.0, ease(pt))

        number_line(img, glow_zero=True)

        # Score cards
        cw,ch,cpad = 195,118,14
        card(img,cpad,16,cw,ch,"Red Marble",
             [f"Pos:  {rp:+.1f}", f"Dist: {abs(rp):.1f}"],
             bg_col=(255,235,235,215),accent=C["RED_DOT"])
        card(img,W-cw-cpad,16,cw,ch,"Blue Marble",
             [f"Pos:  {bp:+.1f}", f"Dist: {bp:.1f}"],
             bg_col=(230,240,255,215),accent=C["BLUE_DOT"])

        # Distance arrows below axis
        if i < race_end:
            aa = int(255*ease(min(1,t*4)))
            zx = nl_to_px(0)
            rx = nl_to_px(rp); bx2 = nl_to_px(bp)
            ay = NL_Y+30
            cr_ = tuple(list(C["RED_DOT"][:3])+[aa])
            cb_ = tuple(list(C["BLUE_DOT"][:3])+[aa])
            if abs(rx-zx)>6:
                d.line([(rx,ay),(zx-2,ay)],fill=cr_,width=3)
                d.polygon([(zx,ay),(zx-8,ay-4),(zx-8,ay+4)],fill=cr_)
            if abs(bx2-zx)>6:
                d.line([(bx2,ay+12),(zx+2,ay+12)],fill=cb_,width=3)
                d.polygon([(zx,ay+12),(zx+8,ay+8),(zx+8,ay+16)],fill=cb_)

        # Bounce when near zero
        bn = 0
        if i >= race_end-FPS//3:
            bt = (i-(race_end-FPS//3))/(FPS//3)
            bn = int(-8*math.sin(bt*math.pi))

        marble(img, rp, C["RED_DOT"],  glow=(i>=race_end-2), bounce=bn)
        marble(img, bp, C["BLUE_DOT"], glow=(i>=race_end-2), bounce=bn)

        # Celebrate
        if i >= race_end:
            ct = (i-race_end)/(n-race_end+1)
            for k in range(nc):
                cfx = int(cx[k]+cdx[k]*ct*55)
                cfy = int(cy[k]+cdy[k]*ct*55+180*ct*ct)
                cfr = int(cr[k])
                ccol = tuple(list(ccols[cci[k]][:3])+[max(0,int(255*(1-ct*1.2)))])
                d.ellipse([cfx-cfr,cfy-cfr,cfx+cfr,cfy+cfr],fill=ccol)
            a_t = int(255*ease(min(1,ct*3)))
            tf  = load_font(C["FONT_BOLD"], int(lerp(120,176,ease_out(min(1,ct*2)))))
            tt  = "It's a TIE!  \U0001F389"
            bb  = d.textbbox((0,0),tt,font=tf)
            tw  = bb[2]-bb[0]
            tx2 = (W-tw)//2
            ty2 = H//2-95
            d.text((tx2+3,ty2+3),tt,font=tf,fill=(255,255,255,a_t))
            d.text((tx2,ty2),tt,font=tf,fill=tuple(list(C["YELLOW_HL"][:3])+[a_t]))
            if ct>0.38:
                a2 = int(255*ease((ct-0.38)/0.62))
                centred(img,"Both reached zero at the same time!",
                        H//2-18,"annot",alpha=a2)
        top_label(img); watermark(img)
        yield img

def gen_reveal(n):
    for i in range(n):
        img = new_frame()
        d   = ImageDraw.Draw(img)
        t   = i/n
        number_line(img, glow_zero=True)
        marble(img, 0.0, C["RED_DOT"],  glow=True, bounce=-3)
        marble(img, 0.0, C["BLUE_DOT"], glow=True)
        centred(img,"Why did they tie?", 22,"sub")

        a1 = int(255*ease(min(1,t*2.5)))
        sx = nl_to_px(C["RED_START"]); ex = nl_to_px(0); br = NL_Y-82
        if a1>10:
            cr_ = tuple(list(C["RED_DOT"][:3])+[a1])
            d.line([(sx,br),(ex,br)],fill=cr_,width=4)
            d.line([(sx,br-8),(sx,br+8)],fill=cr_,width=3)
            d.line([(ex,br-8),(ex,br+8)],fill=cr_,width=3)
            centred(img,"7 steps",(br-36),"annot",
                    col=tuple(C["RED_DOT"]),alpha=a1)

        a2 = int(255*ease(min(1,max(0,t*2.5-0.4))))
        bx = nl_to_px(C["BLUE_START"]); br2 = NL_Y-82
        if a2>10:
            cb_ = tuple(list(C["BLUE_DOT"][:3])+[a2])
            d.line([(ex,br2),(bx,br2)],fill=cb_,width=4)
            d.line([(ex,br2-8),(ex,br2+8)],fill=cb_,width=3)
            d.line([(bx,br2-8),(bx,br2+8)],fill=cb_,width=3)
            centred(img,"7 steps",(br2-36),"annot",
                    col=tuple(C["BLUE_DOT"]),alpha=a2)

        # Expert panel
        a3 = int(255*ease(min(1,max(0,t-0.45)*2.3)))
        if a3>10:
            card(img, W-310,120, 290,255,
                 "For Experts",
                 ["Absolute Value:","  |-7| = 7","  |+7| = 7","","Same distance!","Therefore: Tie."],
                 bg_col=(240,255,240,int(a3*0.92)), accent=C["GREEN_HL"])

        if t>0.62:
            a4 = int(255*ease((t-0.62)/0.38))
            centred(img,"Same distance  =>  Same time!",
                    NL_Y+80,"annot",alpha=a4)
        top_label(img); watermark(img)
        yield img

def gen_summary(n):
    bullets = [
        ("Red   Negative numbers","are LEFT of zero"),
        ("Blue  Positive numbers", "are RIGHT of zero"),
        ("Distance from zero",     "= Absolute Value"),
        ("-7 and +7",              "are equidistant from zero"),
    ]
    for i in range(n):
        img = new_frame()
        t   = i/n
        centred(img,"What we learned!", 18,"sub")
        d = ImageDraw.Draw(img)
        for k,(head,body) in enumerate(bullets):
            at = clamp(t*5 - k*0.85)
            a  = int(255*ease(at))
            if a<5: continue
            bx,by,bw,bh = 75, 108+k*138, W-150, 108
            d.rounded_rectangle([bx+3,by+3,bx+bw+3,by+bh+3],
                                  radius=14,fill=(0,0,0,int(a*0.14)))
            d.rounded_rectangle([bx,by,bx+bw,by+bh],
                                  radius=14,fill=(255,255,255,a),
                                  outline=(210,210,210,a),width=2)
            d.text((bx+18,by+14),head,font=F["card"],fill=(45,45,45,a))
            d.text((bx+18,by+50),body,font=F["card_s"],fill=(45,45,45,a))
        top_label(img); watermark(img)
        yield img

def gen_closing(n):
    PRIMARY = C["BLUE_DOT"][:3]
    for i in range(n):
        img = Image.new("RGBA",(W,H),tuple(PRIMARY)+(255,))
        d   = ImageDraw.Draw(img)
        t   = i/n
        a   = int(255*ease(min(1,t*4)))
        lines = [("Made by Mugambi Ndwiga", F["closing"]),
                 ("@craftsandengineering",   F["closing_s"])]
        y = H//2 - 55
        for txt,f in lines:
            bb = d.textbbox((0,0),txt,font=f)
            tw = bb[2]-bb[0]
            tx = (W-tw)//2
            d.text((tx+2,y+2),txt,font=f,fill=(0,0,60,a//2))
            d.text((tx,y),txt,font=f,fill=(255,255,255,a))
            y += bb[3]-bb[1]+16
        watermark(img)
        yield img

# ── cross-fade iterator ─────────────────────────────────────────────

def cross_fade_frames(fa, fb, n=10):
    for j in range(n):
        t = j/n
        blended = Image.blend(fa.convert("RGBA"), fb.convert("RGBA"), t)
        yield blended

# ── main render loop (streaming to ffmpeg) ─────────────────────────────────────

def render():
    os.makedirs(os.path.dirname(C["OUTPUT_PATH"]), exist_ok=True)

    scene_secs = {
        "hook":    5.0,
        "intro":   6.0,
        "explain": 5.0,
        "count":   3.0,
        "race":    10.0,
        "reveal":  7.0,
        "summary": 5.0,
        "closing": 2.0,
    }
    def sf(s): return max(1, int(round(s*FPS)))

    scene_gens = [
        ("hook",    gen_hook(sf(scene_secs["hook"]))),
        ("intro",   gen_intro(sf(scene_secs["intro"]))),
        ("explain", gen_explain(sf(scene_secs["explain"]))),
        ("count",   gen_countdown(sf(scene_secs["count"]))),
        ("race",    gen_race(sf(scene_secs["race"]))),
        ("reveal",  gen_reveal(sf(scene_secs["reveal"]))),
        ("summary", gen_summary(sf(scene_secs["summary"]))),
        ("closing", gen_closing(sf(scene_secs["closing"]))),
    ]

    ffcmd = [
        "ffmpeg","-y",
        "-f","rawvideo","-vcodec","rawvideo",
        "-s",f"{W}x{H}","-pix_fmt","rgb24",
        "-r",str(FPS),"-i","pipe:0",
        "-c:v","libx264","-crf",str(C["QUALITY_CRF"]),
        "-preset","fast","-pix_fmt","yuv420p",
        "-movflags","+faststart",
        C["OUTPUT_PATH"]
    ]
    proc = subprocess.Popen(ffcmd, stdin=subprocess.PIPE,
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    FADE = 10
    total_written = 0

    # We need to peek at last frame of each scene for the fade
    # Strategy: collect each scene's frames lazily, keeping only last frame
    prev_last = None

    for scene_idx, (name, gen) in enumerate(scene_gens):
        print(f"  Rendering scene: {name} ...", flush=True)
        buf = []   # buffer of at most FADE+1 frames to allow look-back
        last = None

        # If we have a prev_last, emit fade from prev scene to first frame of this
        first_frame = None

        for frame in gen:
            if first_frame is None:
                first_frame = frame
                if prev_last is not None:
                    # emit cross-fade
                    for ff in cross_fade_frames(prev_last, first_frame, FADE):
                        proc.stdin.write(np.array(ff.convert("RGB"),dtype=np.uint8).tobytes())
                        total_written += 1
            # Emit this frame
            proc.stdin.write(np.array(frame.convert("RGB"),dtype=np.uint8).tobytes())
            total_written += 1
            last = frame

        prev_last = last

    proc.stdin.close()
    proc.wait()
    sz = os.path.getsize(C["OUTPUT_PATH"])/1e6
    print(f"\n  Total frames written: {total_written}")
    print(f"  File size: {sz:.2f} MB")
    if proc.returncode != 0:
        raise RuntimeError("ffmpeg encoding failed")
    return C["OUTPUT_PATH"]

if __name__ == "__main__":
    print("Number Line Race – rendering...\n")
    path = render()
    print(f"\n  Video saved to: {path}")

Number Line Race – rendering...

  Rendering scene: hook ...
  Rendering scene: intro ...


/tmp/ipykernel_317/3395125571.py:266: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img.alpha_composite(Image.fromarray(arr,"RGBA"))


  Rendering scene: explain ...
  Rendering scene: count ...
  Rendering scene: race ...
  Rendering scene: reveal ...
  Rendering scene: summary ...
  Rendering scene: closing ...

  Total frames written: 1102
  File size: 0.80 MB

  Video saved to: /mnt/user-data/outputs/number_line_race.mp4


In [5]:
from IPython.display import Video
from google.colab import files

# Note: 'path' contains the result of the last render() call in the main cell.
# To force a fresh render from this cell, uncomment the lines below:
# print("Re-rendering video...")
# path = render()

# Display the video in the notebook
print(f"Displaying: {path}")
display(Video(path, embed=True, width=800))

# Trigger a download of the video file
print(f"Triggering download for: {path}")
files.download(path)

Displaying: /mnt/user-data/outputs/number_line_race.mp4


Triggering download for: /mnt/user-data/outputs/number_line_race.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>